In [1]:
import json

import numpy as np
import scipy
from tabulate import tabulate


In [2]:
sleep_metrics_file_anysleep = "table_s4a_anysleep.json"  # generated by table_s4_anysleep_metrics.ipynb
sleep_metrics_anysleep = json.load(open(sleep_metrics_file_anysleep, "r"))
sleep_metrics_file_usleep = "table_s4a_usleep.json"  # generated by table_s4_usleep_metrics.ipynb
sleep_metrics_usleep = json.load(open(sleep_metrics_file_usleep, "r"))

total_dur_map_anysleep = sleep_metrics_anysleep["total_dur_map"]
total_dur_gt_anysleep = sleep_metrics_anysleep["total_dur_gt"]
total_dur_map_usleep = sleep_metrics_usleep["total_dur_map"]
total_dur_gt_usleep = sleep_metrics_usleep["total_dur_gt"]

In [3]:
def get_avg_std(met_dict, in_hours):
    avg = np.mean(list(met_dict.values()))
    std = np.std(list(met_dict.values()))
    if in_hours:
        avg_neg = avg < 0
        avg_h = int(abs(avg) / 60)
        avg_min = int(np.round(abs(avg) % 60))
        std_h = int(std / 60)
        std_min = int(np.round(std % 60))
        return (f"{'-' if avg_neg else ''}{avg_h}:{str(avg_min).zfill(2)}"
                f"$\\pm${std_h}:{str(std_min).zfill(2)}")
    return f"{avg:.2f}$\\pm${std:.2f}"


def get_spearman(p_met_d, gt_met_d):
    p_agg = {}
    for mr, mr_dict in p_met_d.items():
        for s_id, s_val in mr_dict["1"].items():
            if s_id not in p_agg:
                p_agg[s_id] = []
            p_agg[s_id].append(s_val)

    s_ids = list(sorted(p_agg.keys()))
    p_vals = [np.mean(p_agg[k]) for k in s_ids]
    gt_vals = [gt_met_d[k] for k in s_ids]
    corr, p = scipy.stats.spearmanr(p_vals, gt_vals)
    return f"{corr:.2f}"


pred_total_sleep_time_anysleep = {}
for stage, st_dict in total_dur_map_anysleep.items():
    if stage == "0":
        # Wake is not sleep time
        continue
    for mr, mr_dict in st_dict.items():
        if mr not in pred_total_sleep_time_anysleep:
            pred_total_sleep_time_anysleep[mr] = {"1": {}}
        for s_id, s_val in mr_dict["1"].items():
            if s_id not in pred_total_sleep_time_anysleep[mr]["1"]:
                pred_total_sleep_time_anysleep[mr]["1"][s_id] = 0
            pred_total_sleep_time_anysleep[mr]["1"][s_id] += s_val

gt_total_sleep_time_anysleep = {}
for stage, st_dict in total_dur_gt_anysleep.items():
    if stage == "0":
        # Wake is not sleep time
        continue
    for s_id, s_val in st_dict.items():
        if s_id not in gt_total_sleep_time_anysleep:
            gt_total_sleep_time_anysleep[s_id] = 0
        gt_total_sleep_time_anysleep[s_id] += s_val

pred_total_sleep_time_usleep = {}
for stage, st_dict in total_dur_map_usleep.items():
    if stage == "0":
        # Wake is not sleep time
        continue
    for mr, mr_dict in st_dict.items():
        if mr not in pred_total_sleep_time_usleep:
            pred_total_sleep_time_usleep[mr] = {"1": {}}
        for s_id, s_val in mr_dict["1"].items():
            if s_id not in pred_total_sleep_time_usleep[mr]["1"]:
                pred_total_sleep_time_usleep[mr]["1"][s_id] = 0
            pred_total_sleep_time_usleep[mr]["1"][s_id] += s_val

gt_total_sleep_time_usleep = {}
for stage, st_dict in total_dur_gt_usleep.items():
    if stage == "0":
        # Wake is not sleep time
        continue
    for s_id, s_val in st_dict.items():
        if s_id not in gt_total_sleep_time_usleep:
            gt_total_sleep_time_usleep[s_id] = 0
        gt_total_sleep_time_usleep[s_id] += s_val

table_data = []
for metric_name, p_metric_dict_anysleep, p_metric_dict_usleep, gt_metric_dict in [
    ("Total sleep time", pred_total_sleep_time_anysleep, pred_total_sleep_time_usleep, gt_total_sleep_time_anysleep),
    ("Total Wake duration", total_dur_map_anysleep["0"], total_dur_map_usleep["0"], total_dur_gt_anysleep["0"]),
    ("Total N1 duration", total_dur_map_anysleep["1"], total_dur_map_usleep["1"], total_dur_gt_anysleep["1"]),
    ("Total N2 duration", total_dur_map_anysleep["2"], total_dur_map_usleep["2"], total_dur_gt_anysleep["2"]),
    ("Total N3 duration", total_dur_map_anysleep["3"], total_dur_map_usleep["3"], total_dur_gt_anysleep["3"]),
    ("Total REM duration", total_dur_map_anysleep["4"], total_dur_map_usleep["4"], total_dur_gt_anysleep["4"]),
]:
    print(metric_name)

    # ensure same subjects
    first_mr = list(p_metric_dict_anysleep.keys())[0]
    shared_s_ids = set(p_metric_dict_anysleep[first_mr]["1"].keys()) & set(gt_metric_dict.keys())
    if set(p_metric_dict_anysleep[first_mr]["1"].keys()) != shared_s_ids:
        print(f"Missing keys from pred: {set(p_metric_dict_anysleep[first_mr]['1'].keys()) - shared_s_ids}")
    if set(gt_metric_dict.keys()) != shared_s_ids:
        print(f"Missing keys from ground truth: {set(gt_metric_dict.keys()) - shared_s_ids}")
    # filter to shared subjects
    gt_metric_dict = {k: v for k, v in gt_metric_dict.items() if k in shared_s_ids}
    pred_dict_anysleep = {
        f"{k}_{k2}": v2
        for k, v in p_metric_dict_anysleep.items()
        for k2, v2 in v["1"].items() if k2 in shared_s_ids
    }
    delta_dict_anysleep = {f"{mr}_{s_id}": gt_metric_dict[s_id] - pred_dict_anysleep[f"{mr}_{s_id}"]
                           for s_id in shared_s_ids for mr in p_metric_dict_anysleep}

    # ensure same subjects
    first_mr = list(p_metric_dict_usleep.keys())[0]
    shared_s_ids = set(p_metric_dict_usleep[first_mr]["1"].keys()) & set(gt_metric_dict.keys())
    if set(p_metric_dict_usleep[first_mr]["1"].keys()) != shared_s_ids:
        print(f"Missing keys from pred: {set(p_metric_dict_usleep[first_mr]['1'].keys()) - shared_s_ids}")
    if set(gt_metric_dict.keys()) != shared_s_ids:
        print(f"Missing keys from ground truth: {set(gt_metric_dict.keys()) - shared_s_ids}")
    # filter to shared subjects
    gt_metric_dict = {k: v for k, v in gt_metric_dict.items() if k in shared_s_ids}
    pred_dict_usleep = {
        f"{k}_{k2}": v2
        for k, v in p_metric_dict_usleep.items()
        for k2, v2 in v["1"].items() if k2 in shared_s_ids
    }
    delta_dict_usleep = {f"{mr}_{s_id}": gt_metric_dict[s_id] - pred_dict_usleep[f"{mr}_{s_id}"]
                         for s_id in shared_s_ids for mr in p_metric_dict_usleep}

    table_data.append([
        metric_name,
        get_avg_std(gt_metric_dict, in_hours=True),
        get_avg_std(pred_dict_anysleep, in_hours=True),
        get_avg_std(delta_dict_anysleep, in_hours=True),
        get_spearman(p_metric_dict_anysleep, gt_metric_dict),
        get_avg_std(pred_dict_usleep, in_hours=True),
        get_avg_std(delta_dict_usleep, in_hours=True),
        get_spearman(p_metric_dict_usleep, gt_metric_dict)
    ])

Total sleep time
Total Wake duration
Total N1 duration
Total N2 duration
Total N3 duration
Total REM duration


In [4]:
tabulate(table_data,
         headers=["Metric", "Expert-derived", "AnySleep", "Delta (Exp. - AnySleep)", "Spearman $\\rho$", "U-Sleep",
                  "Delta (Exp. - U-Sleep)",
                  "Spearman $\\rho$"],
         tablefmt="unsafehtml", floatfmt=".2f")

Metric,Expert-derived,AnySleep,Delta (Exp. - AnySleep),Spearman $\rho$,U-Sleep,Delta (Exp. - U-Sleep),Spearman $\rho$
Total sleep time,6:17$\pm$1:14,6:23$\pm$1:13,-0:06$\pm$0:19,0.97,6:23$\pm$1:13,-0:06$\pm$0:18,0.97
Total Wake duration,1:30$\pm$0:59,1:24$\pm$0:57,0:06$\pm$0:19,0.95,1:23$\pm$0:56,0:06$\pm$0:18,0.96
Total N1 duration,0:50$\pm$0:29,0:35$\pm$0:25,0:15$\pm$0:29,0.50,0:29$\pm$0:21,0:21$\pm$0:28,0.48
Total N2 duration,3:12$\pm$1:07,3:18$\pm$0:49,-0:05$\pm$0:47,0.69,3:27$\pm$0:50,-0:15$\pm$0:46,0.71
Total N3 duration,1:05$\pm$0:40,1:13$\pm$0:36,-0:08$\pm$0:36,0.55,1:12$\pm$0:37,-0:07$\pm$0:36,0.56
Total REM duration,1:09$\pm$0:32,1:16$\pm$0:34,-0:07$\pm$0:13,0.93,1:14$\pm$0:34,-0:05$\pm$0:13,0.92


In [5]:
print(tabulate(table_data,
               headers=["Metric", "Expert-derived", "AnySleep", "Delta (Exp. - AnySleep)", "Spearman $\\rho$",
                        "U-Sleep", "Delta (Exp. - U-Sleep)",
                        "Spearman $\\rho$"],
               tablefmt="latex_raw", floatfmt=".2f"))

\begin{tabular}{llllrllr}
\hline
 Metric              & Expert-derived   & AnySleep      & Delta (Exp. - AnySleep)   &   Spearman $\rho$ & U-Sleep       & Delta (Exp. - U-Sleep)   &   Spearman $\rho$ \\
\hline
 Total sleep time    & 6:17$\pm$1:14    & 6:23$\pm$1:13 & -0:06$\pm$0:19            &              0.97 & 6:23$\pm$1:13 & -0:06$\pm$0:18           &              0.97 \\
 Total Wake duration & 1:30$\pm$0:59    & 1:24$\pm$0:57 & 0:06$\pm$0:19             &              0.95 & 1:23$\pm$0:56 & 0:06$\pm$0:18            &              0.96 \\
 Total N1 duration   & 0:50$\pm$0:29    & 0:35$\pm$0:25 & 0:15$\pm$0:29             &              0.50 & 0:29$\pm$0:21 & 0:21$\pm$0:28            &              0.48 \\
 Total N2 duration   & 3:12$\pm$1:07    & 3:18$\pm$0:49 & -0:05$\pm$0:47            &              0.69 & 3:27$\pm$0:50 & -0:15$\pm$0:46           &              0.71 \\
 Total N3 duration   & 1:05$\pm$0:40    & 1:13$\pm$0:36 & -0:08$\pm$0:36            &              0.55 & 1:12